In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)

from processing import SiphraAcquisition
from analysis import fit_peak_expbg


In [2]:
# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used
# Energy emission peaks in MeV
Cs237_MEV = 0.662

In [3]:
#Make sure json files have matching names to the dat files. 

channels = [f'Ch{n}' for n in range(2,17,2)]
print(channels)
files_lsts = {}
files_lsts[channels[0]] = sorted(Path(project_root+'/data/260311').glob('SUBTRACTED*Ch2*.csv'))
files_lsts[channels[0]].pop(3)
files_lsts[channels[0]].pop(3)
files_lsts[channels[0]].pop(3)
for ch in channels[1:]:
    files_lsts[ch] = sorted(Path(project_root+'/data/260311').glob(f'SUBTRACTED*{ch}*.csv'))

vbiases = [28.0, 28.5, 29.0]
# Datasets
datasets = {}
for ch, lst in files_lsts.items():
    datasets[ch] = []
    datasets[ch].append(SiphraAcquisition(lst[2]))
    datasets[ch].append(SiphraAcquisition(lst[0]))
    datasets[ch].append(SiphraAcquisition(lst[1]))

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kCyan, ROOT.kYellow, ROOT.kSpring, ROOT.kViolet]


['Ch2', 'Ch4', 'Ch6', 'Ch8', 'Ch10', 'Ch12', 'Ch14', 'Ch16']


In [4]:
canvas_raw = []
cv_names = ["canvas_raw_"+str(_) for _ in range(1, 9)]
for name in cv_names:
    if ROOT.gROOT.FindObject(name):
        ROOT.gROOT.FindObject(name).Close()
ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.15)

legends_raw = [ROOT.TLegend(0.65, 0.7, 0.8, 0.88) for _ in range(1,9)]
Yinit = 0.82 # For stat boxes

hists_raw_dict = {}
hists_raw_names = ["Ch2", "Ch4", "Ch6", "Ch8", "Ch10", "Ch12", "Ch14", "Ch16"]
voltages = ["28.0V", "28.5V", "29.0V"]

Yinit = 0.82 # For stat boxes

for name, cv, lg in zip(hists_raw_names, cv_names, legends_raw):
    dataset = datasets[name]
    hists_raw_dict[name] = []
    current_hist_list = hists_raw_dict[name]
    canvas_raw.append(ROOT.TCanvas(cv, cv, 800, 600))
    lg.SetHeader('Voltage')
    # legends_raw.append(ROOT.TLegend(0.65, 0.7, 0.8, 0.88))
    for idx, (acq, color) in enumerate(zip(dataset, colors)):
        # print(f"Current file: {acq.filepath.name}")
        ch = acq.active_chs[0]
        hist = ROOT.TH1F(f"{name}, {voltages[idx]}", "", BITS9, 0, BITS12)
        hist.Fill(acq[ch])
        hist.Scale(1/acq.exposure)
        #Preeting up..
        hist.GetXaxis().SetTitle("ADC channel number")
        hist.GetYaxis().SetTitle(r"Count rate (s^{-1})")
        hist.SetLineColor(color+2)
        hist.SetTitle(name)
        lg.AddEntry(hist, f"{voltages[idx]}", 'l')
        current_hist_list.append(hist)
        current_hist_list[-1].Draw('sames hist')
        lg.Draw()
    canvas_raw[-1].SetLogy()
    ROOT.gPad.Update()
    for i, sp in enumerate(current_hist_list):
        st = sp.FindObject("stats")
        # print(type(st))
        st.SetY1NDC(Yinit - i*0.08)
        st.SetY2NDC(0.1)
    #legends_raw[-1].Draw()
    canvas_raw[-1].Draw()